# Blind Spots of Frontier Models
## Model: `tiiuae/Falcon3-3B-Base`

This notebook probes the **blind spots** of the Falcon3-3B-Base model a raw pretrained language model (not finetuned) with 3 billion parameters, released by the Technology Innovation Institute in December 2024.

**Goal:** Find at least 10 diverse inputs where the model makes incorrect predictions, then upload them as a HuggingFace dataset.


## Step 1: Install Dependencies

In [ ]:
!pip install transformers accelerate torch datasets huggingface_hub -q

## Step 2: Load the Model

We use HuggingFace `pipeline` with `bfloat16` precision to fit within the free T4 GPU's 15GB VRAM.

In [ ]:
import torch
from transformers import pipeline

print("Loading Falcon3-3B-Base...")
pipe = pipeline(
    "text-generation",
    model="tiiuae/Falcon3-3B-Base",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print("✅ Model loaded successfully!")

## Step 3: Define the Probing Helper

In [1]:
def probe(prompt, max_new_tokens=60):
    """Run the model on a prompt and return only the new tokens generated."""
    result = pipe(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,          
        repetition_penalty=1.1,
    )
    full_output = result[0]["generated_text"]
    continuation = full_output[len(prompt):].strip()
    print(f"PROMPT   : {prompt}")
    print(f"MODEL OUT: {continuation}")
    print("-" * 70)
    return continuation

## Step 4: Probe the Model 10 Diverse Blind Spot Categories

We test across 10 categories that small base models commonly fail at.

### Category 1: Multi-step Arithmetic

In [ ]:
out1 = probe("What is 17 × 24? Show the calculation and give the final answer:")
# Expected: 408

### Category 2: String / Character Reversal

In [ ]:
out2 = probe("Spell the word 'elephant' backwards. The answer is:")
# Expected: tnahpele

### Category 3: Spatial / Directional Reasoning

In [ ]:
out3 = probe("I am facing north. I turn 90 degrees to my right, then 180 degrees to my left. I am now facing:")
# Expected: West

### Category 4: Date Arithmetic

In [ ]:
out4 = probe("If today is January 20, and I add exactly 50 days, the resulting date is:")
# Expected: March 11

### Category 5: Negation / Exclusion Reasoning

In [ ]:
out5 = probe("Name a country in South America that is NOT Brazil, NOT Argentina, and NOT Colombia:")
# Expected: any valid answer e.g. Chile, Peru, Venezuela...

### Category 6: Obscure Geography

In [ ]:
out6 = probe("The capital city of Burkina Faso is:")
# Expected: Ouagadougou

### Category 7: Uncommon Scientific Facts

In [ ]:
out7 = probe("The chemical symbol for the element Tungsten on the periodic table is:")
# Expected: W

### Category 8: Sorting / Ordering

In [ ]:
out8 = probe("Sort these numbers from smallest to largest: 42, 7, 19, 3, 88, 15. Answer:")
# Expected: 3, 7, 15, 19, 42, 88

### Category 9: Logical Syllogism

In [ ]:
out9 = probe("All glurbs are frizzy. No frizzy things are blue. Therefore, are glurbs blue? Answer yes or no and explain:")
# Expected: No — glurbs are not blue because they are frizzy and no frizzy things are blue.

### Category 10: Cross-lingual / Translation

In [ ]:
out10 = probe("Translate the following sentence into Portuguese: 'The library closes at nine o'clock in the evening.' Translation:")
# Expected: 'A biblioteca fecha às nove horas da noite.'

## Step 5: Collect Results into a Dataset

Edit the `model_output` fields below with what the model actually said in each cell above.

In [ ]:
records = [
    {
        "id": 1,
        "category": "arithmetic",
        "input_prompt": "What is 17 × 24? Show the calculation and give the final answer:",
        "expected_output": "408",
        "model_output": out1,
        "is_correct": "408" in out1,
    },
    {
        "id": 2,
        "category": "string_reversal",
        "input_prompt": "Spell the word 'elephant' backwards. The answer is:",
        "expected_output": "tnahpele",
        "model_output": out2,
        "is_correct": "tnahpele" in out2.lower(),
    },
    {
        "id": 3,
        "category": "spatial_reasoning",
        "input_prompt": "I am facing north. I turn 90 degrees to my right, then 180 degrees to my left. I am now facing:",
        "expected_output": "West",
        "model_output": out3,
        "is_correct": "west" in out3.lower(),
    },
    {
        "id": 4,
        "category": "date_arithmetic",
        "input_prompt": "If today is January 20, and I add exactly 50 days, the resulting date is:",
        "expected_output": "March 11",
        "model_output": out4,
        "is_correct": "march 11" in out4.lower(),
    },
    {
        "id": 5,
        "category": "negation_reasoning",
        "input_prompt": "Name a country in South America that is NOT Brazil, NOT Argentina, and NOT Colombia:",
        "expected_output": "Any valid country e.g. Chile, Peru, Venezuela, Ecuador...",
        "model_output": out5,
        "is_correct": any(c in out5 for c in ["Chile", "Peru", "Venezuela", "Ecuador", "Bolivia", "Uruguay", "Paraguay"]),
    },
    {
        "id": 6,
        "category": "obscure_geography",
        "input_prompt": "The capital city of Burkina Faso is:",
        "expected_output": "Ouagadougou",
        "model_output": out6,
        "is_correct": "ouagadougou" in out6.lower(),
    },
    {
        "id": 7,
        "category": "uncommon_science",
        "input_prompt": "The chemical symbol for the element Tungsten on the periodic table is:",
        "expected_output": "W",
        "model_output": out7,
        "is_correct": " W" in out7 or out7.strip().startswith("W"),
    },
    {
        "id": 8,
        "category": "sorting",
        "input_prompt": "Sort these numbers from smallest to largest: 42, 7, 19, 3, 88, 15. Answer:",
        "expected_output": "3, 7, 15, 19, 42, 88",
        "model_output": out8,
        "is_correct": "3, 7, 15, 19, 42, 88" in out8,
    },
    {
        "id": 9,
        "category": "logical_syllogism",
        "input_prompt": "All glurbs are frizzy. No frizzy things are blue. Therefore, are glurbs blue? Answer yes or no and explain:",
        "expected_output": "No — glurbs are not blue because they are frizzy and no frizzy things are blue.",
        "model_output": out9,
        "is_correct": "no" in out9.lower()[:20],
    },
    {
        "id": 10,
        "category": "translation",
        "input_prompt": "Translate the following sentence into Portuguese: 'The library closes at nine o'clock in the evening.' Translation:",
        "expected_output": "A biblioteca fecha às nove horas da noite.",
        "model_output": out10,
        "is_correct": "biblioteca" in out10.lower(),
    },
]

# Print summary
correct = sum(1 for r in records if r["is_correct"])
print(f"\n📊 Results: {correct}/10 correct ({10 - correct}/10 blind spots found)")
for r in records:
    status = "✅" if r["is_correct"] else "❌"
    print(f"{status} [{r['category']}] Expected: {r['expected_output'][:40]}")

## Step 6: Push Dataset to HuggingFace Hub

You'll need a free HuggingFace account and a **write-access token** from https://huggingface.co/settings/tokens

In [ ]:
from datasets import Dataset
from huggingface_hub import login

login() 

# Convert records to HuggingFace Dataset
ds = Dataset.from_list(records)
print(ds)

HF_USERNAME = "Heiliger44"
DATASET_NAME = "falcon3-3b-blind-spots"

ds.push_to_hub(
    f"{HF_USERNAME}/{DATASET_NAME}",
    private=False, 
)

print(f"\n✅ Dataset published at: https://huggingface.co/datasets/{HF_USERNAME}/{DATASET_NAME}")